# Llamadas a LLMs de Databricks con la API de OpenAI

Databricks Model Serving expone una **API compatible con OpenAI**. Esto significa que puedes usar la librería oficial `openai` (o cualquier herramienta del ecosistema OpenAI) apuntando a tus *serving endpoints* de Databricks, sin cambiar tu código.

## ¿Para qué sirve esto?

- **Portabilidad**: si ya tienes código escrito para OpenAI, solo cambias `api_key` y `base_url` para usar modelos servidos en Databricks (Claude, Llama, GPT-OSS, etc.).
- **Compatibilidad con el ecosistema**: frameworks como LangChain, LlamaIndex o el AI SDK funcionan directamente porque hablan el "dialecto" de la API de OpenAI.
- **Capacidades multimodales**: además de texto, podemos enviar **imágenes** (visión) a modelos que lo soporten, como Claude.

> 💡 Alternativa: también puedes usar el **SDK nativo de Databricks** (`databricks-sdk`) con `w.serving_endpoints.query()`. La API de OpenAI es preferible cuando quieres reutilizar código o integraciones existentes del ecosistema OpenAI.

## 1. Instalación de librerías y utilidades

Instalamos la librería oficial de **OpenAI** para Python, que usaremos como cliente.

- `%pip install openai` instala el paquete en el entorno del notebook.
- Para fijar versión (recomendado en producción): `%pip install openai==1.x.x`.
- Más adelante usaremos también `httpx` y `base64` (vienen incluidos) para enviar imágenes.

In [ ]:
# Instala el cliente oficial de OpenAI para Python
%pip install openai

## 2. Reiniciar el entorno de Python

Tras instalar con `%pip`, reiniciamos el intérprete para que la nueva librería quede disponible.

- `dbutils.library.restartPython()` reinicia solo el proceso de Python (no el clúster).
- ⚠️ Se **pierden todas las variables en memoria**, por eso esta celda va siempre justo después de instalar y antes de empezar a programar.

In [ ]:
# Reinicia el intérprete de Python para cargar la librería recién instalada
dbutils.library.restartPython()

## 3. Crear el cliente de OpenAI apuntando a Databricks

El truco está en configurar dos parámetros del cliente `OpenAI`:

| Parámetro | Valor en Databricks |
|-----------|---------------------|
| `api_key` | Tu **Personal Access Token (PAT)** de Databricks. |
| `base_url` | `https://<tu-workspace>.cloud.databricks.com/serving-endpoints` |

### Cómo obtener el token
En Databricks: **Settings → Developer → Access tokens → Generate new token**.

### Buenas prácticas de seguridad ⚠️
- **Nunca escribas el token en el código.** En el ejemplo aparece como texto solo con fines didácticos.
- Mejor usa **Databricks Secrets**: `dbutils.secrets.get(scope="mi_scope", key="db_token")`.
- O dentro de un notebook puedes obtener el host y el token del contexto automáticamente (ver siguiente celda).

In [ ]:
from openai import OpenAI

# OPCIÓN A (didáctica): credenciales explícitas
# Sustituye por tu token y el hostname de tu workspace.
# client = OpenAI(
#     api_key="YOUR_DATABRICKS_ACCESS_TOKEN",
#     base_url="https://<tu-workspace>.cloud.databricks.com/serving-endpoints"
# )

# OPCIÓN B (recomendada dentro de un notebook): obtener host y token del contexto
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
db_host = "https://" + ctx.tags().get("browserHostName").get()
db_token = ctx.apiToken().get()

client = OpenAI(
    api_key=db_token,
    base_url=f"{db_host}/serving-endpoints"
)

## 4. Enviar una petición de chat simple

Usamos `client.chat.completions.create()`, exactamente igual que con OpenAI.

### Parámetros más usados
| Parámetro | Descripción |
|-----------|-------------|
| `model` | Nombre del *serving endpoint* (no el modelo de OpenAI). |
| `messages` | Lista de mensajes con `role` (`system`/`user`/`assistant`) y `content`. |
| `max_tokens` | Máximo de tokens generados. |
| `temperature` | Creatividad (0 = determinista). |
| `top_p` | Muestreo por núcleo. |
| `stream` | `True` para recibir la respuesta token a token. |
| `stop` | Secuencias que detienen la generación. |

> Nota: `content` puede ser un simple *string* o una **lista de bloques** (`{"type": "text", ...}`), como en este ejemplo. La forma de lista es la que permite mezclar texto e imágenes (multimodal).

In [ ]:
# Petición con la API de OpenAI
completion = client.chat.completions.create(
    model="databricks-claude-sonnet-4-5",
    messages=[
        {
            "role": "system",
            "content": [
                {"type": "text", "text": "Eres Batman, el protector de Gotham City. Responde en español."}
            ]
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "¿Cómo va hoy Gotham City?"}
            ]
        }
    ],
    max_tokens=1024,
    temperature=0.7,
)

# Imprimimos la respuesta del modelo
print(completion.choices[0].message.content)

# Consumo de tokens (útil para controlar costes)
print(f"\nTokens usados -> entrada: {completion.usage.prompt_tokens}, "
      f"salida: {completion.usage.completion_tokens}, total: {completion.usage.total_tokens}")

## 5. Enviar una imagen en una petición de chat (multimodal / visión)

Los modelos con capacidad de **visión** (como Claude) pueden analizar imágenes además de texto. El flujo es:

1. **Descargar** la imagen (aquí con `httpx`).
2. **Codificarla en base64** y construir una *data URL* (`data:image/png;base64,...`).
3. Enviarla como un bloque de tipo `image_url` dentro del `content` del mensaje del usuario, junto al texto.

### Formas de pasar la imagen
- **URL pública directa**: `{"type": "image_url", "image_url": {"url": "https://..."}}`.
- **Base64 incrustado** (lo que hacemos aquí): útil cuando la imagen no es accesible públicamente.

> 💡 Cuidado con el tamaño: las imágenes consumen muchos tokens. Redúcelas si no necesitas el detalle completo.

![interrogation_scene](./Assets/interrogation_scene.png)

In [ ]:
import base64
import httpx

# Descargamos la imagen y la codificamos en base64
image1_url = "https://raw.githubusercontent.com/kuljotSB/DatabricksGenAIEngineer/refs/heads/main/GenAI_Fundamentals/Assets/interrogation_scene.png"
image1_data = base64.standard_b64encode(httpx.get(image1_url).content).decode("utf-8")

# Petición multimodal: texto + imagen
completion = client.chat.completions.create(
    model="databricks-claude-sonnet-4-5",
    messages=[
        {
            "role": "system",
            "content": [
                {"type": "text", "text": "Eres Batman, el Caballero Oscuro y protector de Gotham City. Responde en español."}
            ]
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe esta imagen, por favor."},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{image1_data}"}
                }
            ]
        }
    ],
    max_tokens=1024,
)

# Imprimimos la respuesta final
print(completion.choices[0].message.content)

## 6. (Extra) Respuestas en streaming

Con `stream=True` recibimos la respuesta **token a token** según se genera, en lugar de esperar a que termine. Es ideal para chatbots y mejora la sensación de rapidez (latencia percibida).

In [ ]:
# Petición en streaming: imprimimos los fragmentos según llegan
stream = client.chat.completions.create(
    model="databricks-claude-sonnet-4-5",
    messages=[
        {"role": "system", "content": "Eres Batman. Responde en español."},
        {"role": "user", "content": "Dame un consejo corto para proteger Gotham."}
    ],
    max_tokens=512,
    stream=True,
)

for chunk in stream:
    # Cada chunk trae un fragmento incremental en delta.content
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="")

## Resumen y buenas prácticas

- La **API de OpenAI** funciona contra Databricks cambiando solo `api_key` (tu token) y `base_url` (`.../serving-endpoints`). En `model` pones el **nombre del endpoint**, no un modelo de OpenAI.
- `content` admite **texto simple** o **lista de bloques**; la lista permite **multimodal** (texto + `image_url`).
- Usa `stream=True` para chatbots; revisa `completion.usage` para controlar **costes** por tokens.
- Ajusta `temperature` baja para tareas deterministas y más alta para creatividad.

### Seguridad ⚠️
- No pongas el token en el código. Usa **Databricks Secrets** (`dbutils.secrets.get`) o el contexto del notebook.
- Versiona la librería (`openai==x.y.z`) en producción.

### ¿Qué endpoints existen?
Puedes ver los modelos disponibles en **Serving** en la UI de Databricks, o con el SDK:
```python
from databricks.sdk import WorkspaceClient
for e in WorkspaceClient().serving_endpoints.list():
    print(e.name)
```

### Recursos
- Documentación de Mosaic AI Model Serving (compatibilidad con OpenAI).
- Librería oficial: `openai` (PyPI) y repo `openai/openai-python`.